# Four patches, two heads, one image answer
The same hand-chosen worksheet as the [Vision I lecture](../../vision1.html). The two labels name the two supplied arrangements; this is not a trained general-purpose recognizer.


In [1]:
"""Four visible patches, two attention heads, and one reproducible worksheet.

Parameters are chosen for arithmetic, not fitted. The two named arrangements
are the exercise; this is not a general horizontal/vertical recognizer.
"""
from pathlib import Path
import json
import numpy as np


def parameters():
    root2 = float(np.sqrt(2))
    return {
        'd_model': 4, 'd_k': 2, 'd_v': 2,
        'W_patch': [[.25, 0, 0, 0]] * 4,
        'b_patch': [0, 0, 0, 1],
        'cls': [0, 0, 0, 1],
        'positions': [[0, 0, 0, 0], [0, 0, 0, 0],
                      [0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 1, 0]],
        'heads': [
            {'W_Q': [[0, 0], [0, 0], [0, 0], [1, 1]],
             'W_K': [[root2, 0], [0, root2], [0, 0], [0, 0]],
             'W_V': [[1, 0], [0, 1], [0, 0], [0, 0]]},
            {'W_Q': [[0, 0], [0, 0], [0, 0], [1, 1]],
             'W_K': [[root2, 0], [0, 0], [0, root2], [0, 0]],
             'W_V': [[1, 0], [0, 0], [0, 1], [0, 0]]},
        ],
        'W_O': [[1, 0, 1, 0], [0, 1, 0, 0],
                [-1, 0, 1, 0], [0, 0, 0, 1]],
        'W_class': [[-4, 4], [0, 0], [0, 0], [0, 0]],
        'classes': ['Across the top', 'Down the left'],
        'images': {
            'horizontal': [[1, 1, 1, 1], [1, 1, 1, 1],
                           [0, 0, 0, 0], [0, 0, 0, 0]],
            'vertical': [[1, 1, 0, 0], [1, 1, 0, 0],
                         [1, 1, 0, 0], [1, 1, 0, 0]],
        },
    }


def softmax(x):
    ex = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return ex / ex.sum(axis=-1, keepdims=True)


def forward(image, use_positions=True):
    p = parameters()
    pixels = np.asarray(image, dtype=float)
    # Four row-major patches; each patch's pixels are TL, TR, BL, BR.
    patches = pixels.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)
    content = patches @ np.asarray(p['W_patch']) + p['b_patch']
    E = np.vstack([p['cls'], content])
    if use_positions:
        E = E + p['positions']
    heads = []
    for w in p['heads']:
        Q, K, V = [E @ np.asarray(w[key]) for key in ['W_Q', 'W_K', 'W_V']]
        scores = Q @ K.T / np.sqrt(p['d_k'])
        A = softmax(scores)
        H = A @ V
        heads.append(dict(Q=Q, K=K, V=V, scores=scores, A=A, H=H,
                          contributions=A[0, :, None] * V))
    joined = np.concatenate([h['H'] for h in heads], axis=-1)
    delta = joined @ np.asarray(p['W_O'])
    updated = E + delta
    logits = updated[0] @ np.asarray(p['W_class'])
    probability = softmax(logits)
    return dict(pixels=pixels, patches=patches, content=content, E=E,
                heads=heads, joined=joined, delta=delta, updated=updated,
                logits=logits, probability=probability)


def serializable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {k: serializable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [serializable(v) for v in x]
    return x


def export():
    p = parameters()
    p['cases'] = {
        name + ('_positions' if positions else '_no_positions'):
        serializable(forward(image, positions))
        for name, image in p['images'].items() for positions in [True, False]
    }
    path = Path(__file__).with_name('vision1-worksheet.json')
    path.write_text(json.dumps(p, indent=2) + '\n')
    return p




In [2]:
p = parameters()
r = forward(p['images']['horizontal'])
for name in ['patches', 'content', 'E']:
    print(name, '\n', r[name])


patches 
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
content 
 [[1. 0. 0. 1.]
 [1. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]
E 
 [[0. 0. 0. 1.]
 [1. 0. 0. 1.]
 [1. 0. 1. 1.]
 [0. 1. 0. 1.]
 [0. 1. 1. 1.]]


In [3]:
for i, head in enumerate(r['heads'], 1):
    print('HEAD', i)
    for name in ['Q', 'K', 'V', 'scores', 'A', 'H']:
        print(name, '\n', np.round(head[name], 6))
    print('CLS contributions', np.round(head['contributions'], 6))


HEAD 1
Q 
 [[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]
K 
 [[0.       0.      ]
 [1.414214 0.      ]
 [1.414214 0.      ]
 [0.       1.414214]
 [0.       1.414214]]
V 
 [[0. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]]
scores 
 [[0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]
 [0. 1. 1. 1. 1.]]
A 
 [[0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]
 [0.084224 0.228944 0.228944 0.228944 0.228944]]
H 
 [[0.457888 0.457888]
 [0.457888 0.457888]
 [0.457888 0.457888]
 [0.457888 0.457888]
 [0.457888 0.457888]]
CLS contributions [[0.       0.      ]
 [0.228944 0.      ]
 [0.228944 0.      ]
 [0.       0.228944]
 [0.       0.228944]]
HEAD 2
Q 
 [[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]
K 
 [[0.       0.      ]
 [1.414214 0.      ]
 [1.414214 1.414214]
 [0.       0.      ]
 [0.       1.414214]]
V 
 [[0. 0.]
 [1. 0.]
 [1. 1.]
 [0. 0

In [4]:
for name in ['joined', 'delta', 'updated', 'logits', 'probability']:
    print(name, '\n', np.round(r[name], 6))
print('Loss', -np.log(r['probability'][0]))


joined 
 [[0.457888 0.457888 0.681748 0.681748]
 [0.457888 0.457888 0.681748 0.681748]
 [0.457888 0.457888 0.681748 0.681748]
 [0.457888 0.457888 0.681748 0.681748]
 [0.457888 0.457888 0.681748 0.681748]]
delta 
 [[-0.22386   0.457888  1.139636  0.681748]
 [-0.22386   0.457888  1.139636  0.681748]
 [-0.22386   0.457888  1.139636  0.681748]
 [-0.22386   0.457888  1.139636  0.681748]
 [-0.22386   0.457888  1.139636  0.681748]]
updated 
 [[-0.22386   0.457888  1.139636  1.681748]
 [ 0.77614   0.457888  1.139636  1.681748]
 [ 0.77614   0.457888  2.139636  1.681748]
 [-0.22386   1.457888  1.139636  1.681748]
 [-0.22386   1.457888  2.139636  1.681748]]
logits 
 [ 0.89544 -0.89544]
probability 
 [0.857035 0.142965]
Loss 0.15427637415539527


In [5]:
for name, image in p['images'].items():
    for position in [True, False]:
        print(name, 'positions:', position, forward(image, position)['probability'])


horizontal positions: True [0.85703513 0.14296487]
horizontal positions: False [0.5 0.5]
vertical positions: True [0.14296487 0.85703513]
vertical positions: False [0.5 0.5]
